In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import joblib

# Load preprocessed data
X_train_scaled = np.load("../models/X_train_scaled.npy")
X_test_scaled = np.load("../models/X_test_scaled.npy")
y_train = np.load("../models/y_train.npy")
y_test = np.load("../models/y_test.npy")

print(f"Training data: {X_train_scaled.shape}")
print(f"Test data: {X_test_scaled.shape}")
print(f"Training churn rate: {y_train.mean()*100:.1f}%")
print(f"Test churn rate: {y_test.mean()*100:.1f}%")

Training data: (5625, 30)
Test data: (1407, 30)
Training churn rate: 26.6%
Test churn rate: 26.6%


In [2]:
# Train logistic regression

# Logistic Regression - Simple, interpretable, needs scaled data
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr = lr.predict(X_test_scaled)
y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]

print("LOGISTIC REGRESSION")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_lr):.4f}")

LOGISTIC REGRESSION
Accuracy: 0.8038
Precision: 0.6476
Recall: 0.5749
F1-Score: 0.6091


In [3]:
# Decision Tree - Easy to visualize, no scaling needed
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_scaled, y_train)  # Using scaled data (tree doesn't need it but works)

y_pred_dt = dt.predict(X_test_scaled)

print("DECISION TREE")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_dt):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_dt):.4f}")

DECISION TREE
Accuracy: 0.7164
Precision: 0.4661
Recall: 0.4599
F1-Score: 0.4630


In [4]:
# Random Forest - Best for production, handles complex patterns
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)

print("RANDOM FOREST")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_rf):.4f}")

RANDOM FOREST
Accuracy: 0.7875
Precision: 0.6230
Recall: 0.5080
F1-Score: 0.5596


In [5]:
#compare all models
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_rf)
    ],
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_rf)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_rf)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_rf)
    ]
})

print("MODEL COMPARISON")
print(results.round(4))

MODEL COMPARISON
                 Model  Accuracy  Precision  Recall  F1-Score
0  Logistic Regression    0.8038     0.6476  0.5749    0.6091
1        Decision Tree    0.7164     0.4661  0.4599    0.4630
2        Random Forest    0.7875     0.6230  0.5080    0.5596


In [6]:
# Load feature names
feature_names = joblib.load("../models/feature_names.pkl")

# Get feature importance
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("TOP 10 MOST IMPORTANT FEATURES")
print(importance_df.head(10))

TOP 10 MOST IMPORTANT FEATURES
                           feature  importance
3                     TotalCharges    0.191748
1                           tenure    0.170417
2                   MonthlyCharges    0.168513
10     InternetService_Fiber optic    0.039125
28  PaymentMethod_Electronic check    0.037528
25               Contract_Two year    0.030091
4                      gender_Male    0.029122
13              OnlineSecurity_Yes    0.028514
26            PaperlessBilling_Yes    0.025664
19                 TechSupport_Yes    0.024337


In [7]:
# save best model
# Random Forest is usually best - save it
joblib.dump(rf, "../models/churn_model.pkl")
print("Model saved to ../models/churn_model.pkl")

Model saved to ../models/churn_model.pkl
